In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc 
from sklearn.metrics import f1_score
import json

import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.load_data_gait import *
from src.plot_data_gait import *
from src.train_LSTMGait import train_model, predict_trial

In [2]:
base_path = '../datasets/GAIT_signals/dataset/data'

# Training

In [3]:
results = []
results_path = "../checkpoints/grid_search_results.json"

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

done_keys = {(r["win_size"], r["stride"], r["process"], r["norm"]) for r in results}

for win_size in [50, 100, 200, 400, 500, 700]:
    for overlap in [0, 0.25, 0.5, 0.75]:
        stride = win_size if overlap == 0 else int(win_size * (1 - overlap))

        for process in ["raw", "free_raw"]:
            for norm in ["window", "subj"]:
                key = (win_size, stride, process, norm)
                if key in done_keys:
                    print(f"Skipping already-done combo: win_size={win_size}, stride={stride}, process={process}, norm={norm}")
                    continue

                print(f"Processing with win_size={win_size}, stride={stride}, process={process}, norm={norm}")
                X, y, subjects = load_dataset_gait(base_path, process=process, sensors=['LB'], window_size=win_size, stride=stride)
                X = X.astype(np.float32)
                y = y.astype(np.int64)

                ckpt_path = f"../checkpoints/{process}_win{win_size}_stride{stride}_norm{norm}"
                trained_model, best_val_f1, test_f1 = train_model(X, y, subjects, model_name="cnnbilstm", norm=norm, epochs=200, lr=0.001, patience=20, out_dir=ckpt_path)

                n_subjects = len(np.unique(subjects))
                n_windows = X.shape[0]

                results.append({
                    "win_size": win_size,
                    "stride": stride,
                    "overlap": overlap,
                    "process": process,
                    "norm": norm,
                    "n_subjects": n_subjects,
                    "n_windows": n_windows,
                    "val_f1": round(float(best_val_f1), 6),
                    "test_f1": round(float(test_f1), 6),
                    "checkpoint": ckpt_path,
                })
                print("")
                print("================================")
                print("")

                with open(results_path, 'w') as f:
                    json.dump(results, f, indent=2)

                del X, y, subjects, trained_model
                gc.collect()

print(f"\nDone. {len(results)} combinations trained.")

Processing with win_size=50, stride=50, process=raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 47. Best F1: 0.8901

Val F1: 0.8901  |  Test F1: 0.8643
Checkpoint saved → ../checkpoints/raw_win50_stride50_normwindow\model.pt


Processing with win_size=50, stride=50, process=raw, norm=subj
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 52. Best F1: 0.8866

Val F1: 0.8866  |  Test F1: 0.8659
Checkpoint saved → ../checkpoints/raw_win50_stride50_normsubj\model.pt


Processing with win_size=50, stride=50, process=free_raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 41. Best F1: 0.8876

Val F1: 0.8876  |  Test F1: 0.8568
Checkpoint saved → ../checkpoints/free_raw_win50_stride50_normwindow\model.pt


Processing with win_size=50, stride=50, process=free_raw, norm=subj
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 62085  |  test: 17793
               →  train: 166 subjects (48324 windows)  |  val: 42 subjects (13761 windows)


Early stopping in epoch 59. Best F1: 0.8941

Val F1: 0.8941  |  Test F1: 0.8689
Checkpoint saved → ../checkpoints/free_raw_win50_stride50_normsubj\model.pt


Processing with win_size=50, stride=37, process=raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 83698  |  test: 24004
               →  train: 166 subjects (65155 windows)  |  val: 42 subjects (18543 windows)


...

Early stopping in epoch 43. Best F1: 0.9076

Val F1: 0.9076  |  Test F1: 0.8827
Checkpoint saved → ../checkpoints/free_raw_win700_stride350_normsubj\model.pt


Processing with win_size=700, stride=175, process=raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 14112  |  test: 4187
               →  train: 166 subjects (11004 windows)  |  val: 42 subjects (3108 windows)


Early stopping in epoch 58. Best F1: 0.9115

Val F1: 0.9115  |  Test F1: 0.8854
Checkpoint saved → ../checkpoints/raw_win700_stride175_normwindow\model.pt


Processing with win_size=700, stride=175, process=raw, norm=subj
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 14112  |  test: 4187
               →  train: 166 subjects (11004 windows)  |  val: 42 subjects (3108 windows)


Early stopping in epoch 48. Best F1: 0.8987

Val F1: 0.8987  |  Test F1: 0.8717
Checkpoint saved → ../checkpoints/raw_win700_stride175_normsubj\model.pt


Processing with win_size=700, stride=175, process=free_raw, norm=window
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 14112  |  test: 4187
               →  train: 166 subjects (11004 windows)  |  val: 42 subjects (3108 windows)


Early stopping in epoch 66. Best F1: 0.9124

Val F1: 0.9124  |  Test F1: 0.8856
Checkpoint saved → ../checkpoints/free_raw_win700_stride175_normwindow\model.pt


Processing with win_size=700, stride=175, process=free_raw, norm=subj
Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260
Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 14112  |  test: 4187
               →  train: 166 subjects (11004 windows)  |  val: 42 subjects (3108 windows)


Early stopping in epoch 37. Best F1: 0.9051

Val F1: 0.9051  |  Test F1: 0.8727
Checkpoint saved → ../checkpoints/free_raw_win700_stride175_normsubj\model.pt



Done. 96 combinations trained.


# Results comparison

In [4]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_f1", ascending=False).reset_index(drop=True)
results_df

,win_size,stride,overlap,process,norm,n_subjects,n_windows,val_f1,test_f1,checkpoint
0,400,100,0.75,raw,window,260,35562,0.917324,0.888501,../checkpoints/raw_win400_stride100_normwindow
1,400,200,0.50,raw,window,260,18126,0.917086,0.887998,../checkpoints/raw_win400_stride200_normwindow
2,200,50,0.75,raw,window,260,75840,0.916827,0.895212,../checkpoints/raw_win200_stride50_normwindow
3,200,50,0.75,free_raw,window,260,75840,0.916700,0.894765,../checkpoints/free_raw_win200_stride50_normwi...
4,400,300,0.25,free_raw,window,260,12298,0.916178,0.890164,../checkpoints/free_raw_win400_stride300_normw...
...,...,...,...,...,...,...,...,...,...,...
91,50,37,0.25,free_raw,window,260,107702,0.893584,0.866566,../checkpoints/free_raw_win50_stride37_normwindow
92,50,37,0.25,raw,subj,260,107702,0.892107,0.862056,../checkpoints/raw_win50_stride37_normsubj
93,50,50,0.00,raw,window,260,79878,0.890124,0.864260,../checkpoints/raw_win50_stride50_normwindow
94,50,50,0.00,free_raw,window,260,79878,0.887560,0.856792,../checkpoints/free_raw_win50_stride50_normwindow


In [5]:
best_row = results_df.iloc[0]
print("Best configuration (selected by val_f1):")
print(best_row)
print(f"\n--> Reported test_f1 for this configuration: {best_row['test_f1']:.4f}")

Best configuration (selected by val_f1):
win_size                                                 400
stride                                                   100
overlap                                                 0.75
process                                                  raw
norm                                                  window
n_subjects                                               260
n_windows                                              35562
val_f1                                              0.917324
test_f1                                             0.888501
checkpoint    ../checkpoints/raw_win400_stride100_normwindow
Name: 0, dtype: object

--> Reported test_f1 for this configuration: 0.8885


In [6]:
pivot = results_df.pivot_table(index=["win_size", "stride"], columns="process", values="val_f1")
pivot["free_raw - raw"] = pivot["free_raw"] - pivot["raw"]
pivot.sort_values("free_raw - raw", ascending=False)

process          free_raw       raw  free_raw - raw
win_size stride                                    
500      375     0.911509  0.904127        0.007382
         125     0.912587  0.905335        0.007252
         250     0.912326  0.905466        0.006859
700      350     0.909711  0.903652        0.006059
400      300     0.913775  0.908155        0.005620
700      700     0.910328  0.905241        0.005087
         525     0.909379  0.904539        0.004840
500      500     0.911778  0.907301        0.004477
200      200     0.909722  0.905349        0.004373
         100     0.911338  0.907202        0.004136
400      200     0.912831  0.909042        0.003789
700      175     0.908726  0.905131        0.003595
200      150     0.911917  0.908645        0.003273
50       50      0.890850  0.888378        0.002472
100      100     0.904141  0.901900        0.002240
         50      0.907694  0.905668        0.002026
200      50      0.913552  0.911698        0.001855
100      75      0.907343  0.905699        0.001644
400      400     0.910743  0.909300        0.001443
50       37      0.894304  0.892886        0.001418
100      25      0.910428  0.909041        0.001387
50       12      0.899727  0.898710        0.001016
400      100     0.911145  0.910646        0.000499
50       25      0.896454  0.896333        0.000120

In [7]:
subject_sets = {}
for r in results:
    ckpt_file = os.path.join(r["checkpoint"], "model.pt")
    if not os.path.exists(ckpt_file):
        continue
    import torch
    ckpt = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    subject_sets[(r["win_size"], r["stride"], r["process"])] = set(ckpt["test_subjects"])

reference_key, reference_set = next(iter(subject_sets.items()))
mismatches = {k: v for k, v in subject_sets.items() if v != reference_set}

if mismatches:
    print(f"WARNING: {len(mismatches)} configuration(s) have a different test-subject split than {reference_key}.")
    print("This means n_subjects differs across configs (likely due to short trials being dropped")
    print("at larger window_size), so val_f1/test_f1 are not perfectly comparable across all rows.")
    for k in list(mismatches.keys())[:10]:
        print("  -", k)
else:
    print("OK: all configurations share the exact same test-subject split.")

OK: all configurations share the exact same test-subject split.
